# MeshSelectionSet — Post-Mesh Selection Demo

Demonstrates `g.mesh_selection` — the post-mesh selection system complementary to `g.physical`.

**Physical Groups** operate on *geometry* (pre-mesh): surfaces, curves, points.
**Mesh Selections** operate on *mesh entities* (post-mesh): nodes and elements, using spatial queries.

Both use the same `(dim, tag) + name` identity contract and produce identical output shapes
(`get_nodes()` → `{'tags', 'coords'}`, `get_elements()` → `{'element_ids', 'connectivity'}`).

In [ ]:
from pyGmsh import pyGmsh
import numpy as np

## 1. Geometry & Mesh

Create a simple L-shaped plate and mesh it.

In [ ]:
g = pyGmsh(model_name="MeshSelectionDemo", verbose=True)
g.begin()

# L-shaped plate: two rectangles
lc = 0.5  # mesh size

# Bottom rectangle: 10 x 4
p1 = g.model.add_point(0, 0, 0, lc=lc)
p2 = g.model.add_point(10, 0, 0, lc=lc)
p3 = g.model.add_point(10, 4, 0, lc=lc)
p4 = g.model.add_point(4, 4, 0, lc=lc)
p5 = g.model.add_point(4, 8, 0, lc=lc)
p6 = g.model.add_point(0, 8, 0, lc=lc)

l1 = g.model.add_line(p1, p2)
l2 = g.model.add_line(p2, p3)
l3 = g.model.add_line(p3, p4)
l4 = g.model.add_line(p4, p5)
l5 = g.model.add_line(p5, p6)
l6 = g.model.add_line(p6, p1)

loop = g.model.add_curve_loop([l1, l2, l3, l4, l5, l6])
plate = g.model.add_plane_surface(loop, label="plate")

# Physical groups (pre-mesh, on geometry)
g.physical.add(1, [l1], name="bottom_edge")
g.physical.add(1, [l6], name="left_edge")
g.physical.add(2, [plate], name="plate")

print("\nPhysical Groups:")
g.physical.summary()

In [ ]:
# Generate mesh
g.mesh.generate(2)
g.mesh.renumber_mesh(method='rcm')

print(f"Mesh: {g.mesh.get_nodes()['tags'].shape[0]} nodes")

## 2. Mesh Selections — Spatial Queries

Now that the mesh exists, we can create selections based on *node coordinates* and *element centroids*.

In [ ]:
# Select nodes on the bottom edge (y=0)
g.mesh_selection.add_nodes(
    on_plane=("y", 0.0, 1e-3),
    name="bottom_nodes",
)

# Select nodes on the left edge (x=0)
g.mesh_selection.add_nodes(
    on_plane=("x", 0.0, 1e-3),
    name="left_nodes",
)

# Select a single node nearest to a point (corner load)
g.mesh_selection.add_nodes(
    nearest_to=(10.0, 4.0, 0.0),
    count=1,
    name="corner_load",
)

# Select nodes in a rectangular region
g.mesh_selection.add_nodes(
    in_box=[3, 3, -1, 5, 5, 1],
    name="center_region_nodes",
)

# Select nodes within a sphere (circular zone)
g.mesh_selection.add_nodes(
    in_sphere=(2.0, 2.0, 0.0, 1.5),
    name="circular_zone",
)

print("Node selections:")
g.mesh_selection.summary()

In [ ]:
# Select elements in a bounding box (by centroid)
g.mesh_selection.add_elements(
    dim=2,
    in_box=[0, 0, -1, 4, 4, 1],
    name="bottom_left_elements",
)

# Select elements on a plane (all nodes on y=0)
g.mesh_selection.add_elements(
    dim=2,
    on_plane=("y", 0.0, 0.1),
    name="bottom_row_elements",
)

print("\nAll selections (nodes + elements):")
g.mesh_selection.summary()

## 3. Set Algebra

Combine selections with union, intersection, and difference.

In [ ]:
# Intersection: nodes that are on BOTH bottom and left edges (the corner node)
tag_bottom = g.mesh_selection.get_tag(0, "bottom_nodes")
tag_left = g.mesh_selection.get_tag(0, "left_nodes")

g.mesh_selection.intersection(
    dim=0, tag_a=tag_bottom, tag_b=tag_left,
    name="corner_node",
)

# Union: all boundary nodes (bottom + left)
g.mesh_selection.union(
    dim=0, tag_a=tag_bottom, tag_b=tag_left,
    name="all_boundary_nodes",
)

# Difference: bottom nodes minus the corner
tag_corner = g.mesh_selection.get_tag(0, "corner_node")
g.mesh_selection.difference(
    dim=0, tag_a=tag_bottom, tag_b=tag_corner,
    name="bottom_no_corner",
)

print("After set algebra:")
g.mesh_selection.summary()

## 4. Bridge from Physical Groups

Import an existing physical group as a mesh selection set.

In [ ]:
# Import the "bottom_edge" physical group as a node selection
g.mesh_selection.from_physical(
    dim=1,
    name_or_tag="bottom_edge",
    ms_name="from_pg_bottom",
)

print("After bridge:")
g.mesh_selection.summary()

## 5. Query the Data

The `get_nodes()` and `get_elements()` methods return the **same dict shape** as `PhysicalGroupSet`.

In [ ]:
# Query node set
bottom = g.mesh_selection.get_nodes(0, tag_bottom)
print(f"bottom_nodes: {len(bottom['tags'])} nodes")
print(f"  tags shape:   {bottom['tags'].shape}")
print(f"  coords shape: {bottom['coords'].shape}")
print(f"  Y range: [{bottom['coords'][:, 1].min():.4f}, {bottom['coords'][:, 1].max():.4f}]")
print()

# Query element set
tag_bl = g.mesh_selection.get_tag(2, "bottom_left_elements")
elems = g.mesh_selection.get_elements(2, tag_bl)
print(f"bottom_left_elements: {len(elems['element_ids'])} elements")
print(f"  element_ids shape:  {elems['element_ids'].shape}")
print(f"  connectivity shape: {elems['connectivity'].shape}")
print()

# Compare: same shape as physical group output
pg_bottom = g.physical.get_nodes(1, 1)
print("Physical group 'bottom_edge':")
print(f"  tags shape:   {pg_bottom['tags'].shape}")
print(f"  coords shape: {pg_bottom['coords'].shape}")

## 6. FEMData Integration

Both physical groups and mesh selections are captured as parallel snapshots in `FEMData`.

In [ ]:
fem = g.mesh.get_fem_data(dim=2)

print("=== fem.physical ===")
print(fem.physical.summary())
print()

print("=== fem.mesh_selection ===")
print(fem.mesh_selection.summary())
print()

# Both return identical dict shapes
pg_nodes = fem.physical.get_nodes(1, 1)    # from physical groups
ms_nodes = fem.mesh_selection.get_nodes(0, tag_bottom)  # from mesh selections

print(f"Physical group nodes: {pg_nodes['tags'].shape}")
print(f"Mesh selection nodes: {ms_nodes['tags'].shape}")
print(f"Same dict keys: {set(pg_nodes.keys()) == set(ms_nodes.keys())}")

## 7. Custom Predicate Filter

For complex selection criteria, pass a custom function.

In [ ]:
# Select nodes above the diagonal line y = x
g.mesh_selection.add_nodes(
    predicate=lambda coords: coords[:, 1] > coords[:, 0],
    name="above_diagonal",
)

above = g.mesh_selection.get_nodes(0, g.mesh_selection.get_tag(0, "above_diagonal"))
print(f"Nodes above diagonal y>x: {len(above['tags'])}")
print(f"  All satisfy y>x: {np.all(above['coords'][:, 1] > above['coords'][:, 0])}")

## 8. Visualization

View the mesh and selected regions.

In [ ]:
# View the model geometry
g.model.viewer()

In [ ]:
# View the mesh
g.mesh.viewer()

## Summary

| Feature | `g.physical` | `g.mesh_selection` |
|---------|-------------|-------------------|
| **Operates on** | Geometry (pre-mesh) | Mesh nodes/elements (post-mesh) |
| **Identity** | `(dim, tag) + name` | `(dim, tag) + name` |
| **Creation** | `add(dim, [geom_tags])` | `add_nodes(on_plane=...)`, `add_elements(in_box=...)` |
| **Set algebra** | — | `union`, `intersection`, `difference` |
| **FEMData** | `fem.physical` | `fem.mesh_selection` |
| **Output shape** | `{'tags', 'coords'}` | `{'tags', 'coords'}` (identical) |